In [ ]:
!pip install --upgrade pip
!pip install transformers datasets peft bitsandbytes accelerate evaluate
!pip install -q rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 142.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 

In [ ]:
import math
import numpy as np
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate, torch, random
import json, os, datetime, torch, gc, pathlib
from peft import PeftModel
from contextlib import nullcontext
from tqdm.auto import tqdm
from typing import List, Dict, Any
from transformers import (AutoTokenizer, AutoModelForCausalLM,TrainingArguments, Trainer, set_seed)
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from textwrap import fill
from IPython.display import display, Markdown
from google.colab import files
uploaded = files.upload()

Saving BioASQ-training13b.zip to BioASQ-training13b.zip


In [ ]:
zip_name = list(uploaded.keys())[0]
os.makedirs("/content/data/bioasq", exist_ok=True)
!unzip -q "$zip_name" -d /content/data/bioasq

!ls -l /content/data/bioasq | head

total 4
drwxrwxr-x 2 root root 4096 Oct  7  2024 BioASQ-training13b


In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-11B-Vision-Instruct"
DATA_PATH = "data/bioasq/BioASQ-training13b/training13b.json"
OUTPUT_DIR = "./lora_bioasq_outputs"
ADAPTER_NAME = "lora-bioasq"
MAX_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
SEED = 42
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
#Load Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Pad token ID: {tokenizer.pad_token_id}, EOS token ID: {tokenizer.eos_token_id}")

# Try to load the model in bfloat16 on GPU
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
except Exception as e:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
model.eval()

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Pad token ID: 128004, EOS token ID: 128009


config.json:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

...

model-00003-of-00005.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

MllamaForCausalLM(
  (model): MllamaTextModel(
    (embed_tokens): Embedding(128264, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-2): 3 x MllamaSelfAttentionDecoderLayer(
        (self_attn): MllamaTextSelfAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MllamaTextMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MllamaTextRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MllamaTextRMSNorm((4096,), eps=1e-05)
      )
 

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
with open(DATA_PATH) as f:
    raw_json = json.load(f)

questions = raw_json["questions"]

#Normalize types
def normalize(q):
    ideal = q.get("ideal_answer", "")
    if isinstance(ideal, list):
        ideal = " ".join(ideal).strip()
    q["ideal_answer"] = ideal

    return q

norm_questions = [normalise(q) for q in questions]

# Build HF Dataset
dataset = Dataset.from_list(norm_questions)
print("Loaded", len(dataset), "examples")


dataset = dataset.shuffle(seed=42)
dev_size = int(0.05 * len(dataset))
train_dataset = dataset.select(range(dev_size, len(dataset)))
dev_dataset   = dataset.select(range(dev_size))

Loaded 5389 examples


In [ ]:
def preprocess_example(ex):
    """
    Given a raw example, filter yes/no and format into prompt-target.
    """
    # BioASQ format hasfields "type", "body", "ideal_answer", "snippets"
    q_type = ex.get("type", "").lower()
    if q_type == "yesno" or ex.get("ideal_answer", "").strip().lower() in ["yes", "no"]:
        return None  # skip yes/no
    question = ex.get("body", "").strip()
    ideal = ex.get("ideal_answer", "")
    # If ideal_answer is list, join them; if string, use directly
    if isinstance(ideal, list):
        # join multiple ideal answers into one string
        ideal = " ".join([ans.strip() for ans in ideal if ans.strip()])
    ideal = ideal.strip()
    if not question or not ideal:
        return None
    # Get snippet texts
    snippets = ex.get("snippets", [])
    snippet_texts = []
    for snip in snippets:
        text = snip.get("text", "").strip()
        if text:
            snippet_texts.append(text)
    # Format context bullets
    context_lines = [f"- {s}" for s in snippet_texts]
    prompt = "User: " + question + "\nContext:\n" + "\n".join(context_lines) + "\nAssistant:"
    return {"input": prompt, "target": ideal}

processed = []
for item in train_dataset:
    out = preprocess_example(item)
    if out:
        processed.append(out)

print(f"Total examples after filtering: {len(processed)}")
random.shuffle(processed)

Total examples after filtering: 3751


In [ ]:
def tokenize_and_format(example: Dict[str,str]):
    """
    Tokenizes one prompt-target example into input_ids and labels.
    """
    # Tokenize prompt (User+Context+Assistant prompt)
    tokenized_input = tokenizer(
        example["input"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False
    )
    # Tokenize target (the ideal answer)
    tokenized_target = tokenizer(
        example["target"],
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False
    )
    # Combine input and target ids
    input_ids = tokenized_input["input_ids"] + tokenized_target["input_ids"]

    labels = [-100] * len(tokenized_input["input_ids"]) + tokenized_target["input_ids"]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

# Tokenize all examples
tokenized_data = [tokenize_and_format(ex) for ex in processed if ex]

class DataCollatorForCausalLM:

    """
    Collate function to pad input_ids, labels, and attention_masks for a batch.
    """
    def __init__(self, pad_token_id: int, label_pad_token_id: int = -100):
        self.pad_token_id = pad_token_id
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_size = len(features)
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids_padded = torch.full((batch_size, max_len), self.pad_token_id, dtype=torch.long)
        attention_mask_padded = torch.zeros((batch_size, max_len), dtype=torch.long)
        labels_padded = torch.full((batch_size, max_len), self.label_pad_token_id, dtype=torch.long)
        for i, f in enumerate(features):
            seq_len = len(f["input_ids"])
            input_ids_padded[i, :seq_len] = torch.tensor(f["input_ids"], dtype=torch.long)
            attention_mask_padded[i, :seq_len] = torch.tensor(f["attention_mask"], dtype=torch.long)
            labels_padded[i, :seq_len] = torch.tensor(f["labels"], dtype=torch.long)
        return {"input_ids": input_ids_padded,
                "attention_mask": attention_mask_padded,
                "labels": labels_padded}

collator = DataCollatorForCausalLM(
    pad_token_id=tokenizer.pad_token_id,
    label_pad_token_id=-100
)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    bf16=True,
    logging_strategy="steps",
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)


In [ ]:
train_dataset = tokenized_data

#4‑bit QLoRA config with CPU off‑load
bnb_config4 = BitsAndBytesConfig(
    load_in_4bit        = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload = True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config4,
    device_map          = "auto",
    trust_remote_code   = True
)


model.gradient_checkpointing_enable()
model.config.use_cache = False

#attach LoRA adapters
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none", task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)  # sets requires_grad
model = get_peft_model(model, lora_config)


#cut seq len to 1024
MAX_LENGTH = 1024


# smaller batch
training_args = TrainingArguments(
    output_dir               = OUTPUT_DIR,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 16,
    learning_rate            = 2e-4,
    num_train_epochs         = 3,
    bf16                     = True,
    logging_steps            = 50,
    save_steps               = 500,
    save_total_limit         = 2,
    report_to                = "none",
    remove_unused_columns    = False
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    data_collator   = collator
)

trainer.train()

Loading 4‑bit model …


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,0.586200
100,0.507900
150,0.584300
200,0.570200
250,0.519100
300,0.522800
350,0.493300
400,0.477400
450,0.488100
500,0.473800


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=705, training_loss=0.49805075395191817, metrics={'train_runtime': 7331.8431, 'train_samples_per_second': 1.535, 'train_steps_per_second': 0.096, 'total_flos': 4.723505318247604e+17, 'train_loss': 0.49805075395191817, 'epoch': 3.0})

In [ ]:
model.eval()
n_eval = 3                         # num to sample
eval_examples = random.sample(processed, n_eval)

rouge  = evaluate.load("rouge")
preds, refs = [], []

with torch.no_grad():
    for ex in eval_examples:
        prompt = ex["input"]
        inputs = tokenizer(prompt, return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)

        gen_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0
        )
        # strip prompt tokens
        gen_text = tokenizer.decode(
            gen_ids[0][inputs["input_ids"].size(1):],
            skip_special_tokens=True
        ).strip()

        preds.append(gen_text)
        refs.append(ex["target"])


        print("Q:", prompt.split("\n")[0][6:])
        print("generated:", gen_text[:200] + ("…" if len(gen_text) > 200 else ""))
        print("gold:",      ex["target"][:200] + ("…" if len(ex["target"]) > 200 else ""))
        print("-"*60)

# ROUGE‑L / ROUGE‑2
scores = rouge.compute(predictions=preds, references=refs)
print({k: round(v, 4) for k, v in scores.items()})

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: Is there any genetic determinant of hair pigmentation that could be useful in forensic analyses?
generated: Human hair color is predictable from DNA variants with similarly high accuracies. Several key pigmentation genes have been characterised, in particular the melanocortin 1 receptor gene (MC1R). Here, t…
gold: Yes, there are at least 12 genes associated with human hair color variation such as: TYR, TYRP1, OCA2, SLC45A2, SLC24A5, MC1R, ASIP and KITLG.
------------------------------------------------------------


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: What does polyadenylate-binding protein 4 (PABP4) bind to?
generated: Poly(A) binding protein 4 (PABPC4) binds to poly(A) tails of mRNA. Poly(A) binding protein 4 (PABPC4) binds to poly(A) tails of mRNA. Poly(A) binding protein 4 (PABPC4) binds to poly(A) tails of mRNA.…
gold: PABP4  binds mRNA poly(A) tails.
------------------------------------------------------------
Q: What is the function of GFRAL?
generated: GDNF-family receptor α-like (GFRAL), an orphan member of the GFR-α family, is a high-affinity receptor for GDF15. GFRAL expression was limited to hindbrain neurons and not present in peripheral tissue…
gold: GFRAL (orphan receptor of the glial-derived neurotrophic factor (GDNF) receptor α family) is a high-affinity receptor for GDF15.
GFRAL expression is limited to hindbrain neurons and not present in per…
------------------------------------------------------------
{'rouge1': np.float64(0.3489), 'rouge2': np.float64(0.2214), 'rougeL': np.float64(0.291), 'rougeLsum': np.flo

In [ ]:
adapter_dir = os.path.join(OUTPUT_DIR, ADAPTER_NAME)
os.makedirs(adapter_dir, exist_ok=True)

if isinstance(model, PeftModel):
    model.save_pretrained(adapter_dir)
    print("Saved LoRA adapter to:", adapter_dir)

tokenizer.save_pretrained(adapter_dir)

with open(os.path.join(adapter_dir, "training_args.json"), "w") as f:
    json.dump(training_args.to_dict(), f, indent=2)

meta = {
    "base_model": MODEL_NAME,
    "saved_at": datetime.datetime.utcnow().isoformat() + "Z",
    "context_length": MAX_LENGTH,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "quantization": "4bit_nf4" if training_args.bf16 else "bf16",
}
with open(os.path.join(adapter_dir, "README.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Adapter + tokenizer + configs saved to → {adapter_dir}")

Saved LoRA adapter to: ./lora_bioasq_outputs/lora-bioasq
Adapter + tokenizer + configs saved to → ./lora_bioasq_outputs/lora-bioasq


In [ ]:
!ls -lh /content | grep -i "\.zip$"

files.download("/content/lora-bioasq.zip")

-rw-r--r-- 1 root root 6.7M Jul 21 15:46 BioASQ-training13b.zip
-rw-r--r-- 1 root root  58M Jul 21 18:16 lora-bioasq.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
Compare base vs LoRA on the same eval split using single PeftModel.
"""
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_EVAL = 100
MAX_LENGTH = 1024                   # context window


assert isinstance(model, PeftModel),

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# sample evaluation set
eval_examples = random.sample(processed, N_EVAL)
refs = [ex["target"] for ex in eval_examples]

@torch.inference_mode()
def run_generation(m):
    preds = []
    for ex in tqdm(eval_examples):
        inputs = tokenizer(
            ex["input"],
            return_tensors="pt",
            truncation=True, max_length=MAX_LENGTH
        ).to(DEVICE)
        ids = m.generate(**inputs, max_new_tokens=128,
                         do_sample=False, temperature=0.0)
        preds.append(
            tokenizer.decode(
                ids[0][inputs["input_ids"].size(1):],  # strip prompt
                skip_special_tokens=True
            ).strip()
        )
        torch.cuda.empty_cache()
    return preds

# base performance
with model.disable_adapter():         # LoRA weights inactive
    base_preds = run_generation(model)

# LoRA‑fine‑tuned performance
lora_preds = run_generation(model)

#compute metrics
rouge = evaluate.load("rouge")
bleu  = evaluate.load("sacrebleu")

def short(d): return {k: round(v,3) for k,v in d.items()}

print("\nROUGE (base):", short(rouge.compute(predictions=base_preds,  references=refs)))
print("ROUGE (LoRA):",  short(rouge.compute(predictions=lora_preds,  references=refs)))

base_bleu = bleu.compute(predictions=base_preds, references=[[r] for r in refs])["score"]
lora_bleu = bleu.compute(predictions=lora_preds, references=[[r] for r in refs])["score"]

print("\nBLEU‑4 (base):", round(base_bleu,2))
print("BLEU‑4 (LoRA):",  round(lora_bleu,2))

  0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [ ]:
def wrap(txt, width=120):
    return fill(txt, width=width, replace_whitespace=False)

@torch.inference_mode()
def answer(model, prompt, max_new=128):
    t = tokenizer(prompt, return_tensors="pt",
                  truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    out = model.generate(**t, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][t["input_ids"].size(1):],
                            skip_special_tokens=True).strip()

def inspect(idx):
    ex = eval_examples[idx]
    q  = ex["input"].split("\n")[0][6:]

    # Gold
    gold = ex["target"]

    # Base answer
    with model.disable_adapter():
        base_ans = answer(model, ex["input"])

    # LoRA answer
    lora_ans = answer(model, ex["input"])

    display(Markdown(f"""
**🔹 Question #{idx}**
{wrap(q)}

> **Gold:**
> {wrap(gold)}

| | Answer |
|---|---|
| **Base** | {wrap(base_ans)} |
| **LoRA** | {wrap(lora_ans)} |
"""))

# loop
print(f"Ready. Valid indices: 0 – {N_EVAL-1}.  ENTER = next, q = quit.")
i = 0
while True:
    s = input(f"Inspect index [{i}]: ").strip()
    if s.lower() in {"q", "quit", "exit"}:
        break
    if s:
        try:
            i = int(s)
        except ValueError:
            print("Please enter a number or 'q'."); continue
    if i < 0 or i >= N_EVAL:
        print("Out of range."); continue
    inspect(i)
    i = (i + 1) % N_EVAL